# 1、导入库并创建测试文档

In [ ]:
# 加载文档（读取文件内容） TextLoader 专门用来读取 .txt 文本文件，
# 把其中的文字提取出来，变成 LangChain 能处理的 Document 对象（包含文本内容和元数据）
from langchain_community.document_loaders import TextLoader
# 把长文本切分成小段（chunk）
from langchain.text_splitter import CharacterTextSplitter
# 使用 HuggingFace 的嵌入模型
from langchain_community.embeddings import HuggingFaceEmbeddings
# 使用 FAISS 作为向量存储
from langchain_community.vectorstores import FAISS
# 使用 DeepSeek 的 LLM
from langchain_deepseek import ChatDeepSeek
# 使用 RetrievalQA 链
from langchain.chains import RetrievalQA
import os
from dotenv import load_dotenv  # 新增导入

# 加载 .env 文件中的环境变量
load_dotenv()

True

# 2、准备一个测试文档

In [2]:
# --- 第一步：准备一个测试文档 ---
# 为了让程序立刻能跑起来，我们在代码中直接创建一个小文本
test_text = """
人工智能（AI）是计算机科学的一个分支，致力于创建能够执行通常需要人类智能的任务的系统。
这些任务包括学习、推理、问题解决、感知和语言理解。
机器学习是AI的一个子领域，它使计算机无需明确编程即可学习。
深度学习是机器学习的一个子领域，它使用多层神经网络来分析数据的各个方面。
"""
# 将这个文本保存成一个临时的txt文件
with open("test_doc.txt", "w", encoding="utf-8") as f:
    f.write(test_text)

# 3、加载文档并分块

In [3]:
# --- 第二步：加载文档并分块 ---
# 用LangChain提供的加载器读取txt文件
loader = TextLoader("test_doc.txt", encoding="utf-8")
documents = loader.load()
# 将长文本切成小块，chunk_size=100表示每块大约100个字符
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
print(f"文档已被切分成 {len(docs)} 个小块。")

文档已被切分成 1 个小块。


# 4、构建本地向量库 (FAISS)

In [4]:
# --- 第三步：构建本地向量库 (FAISS) ---
# 使用一个本地的、免费的中文向量模型把文字转为坐标
embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")
# 将文本块存入FAISS向量库，并保存到本地
db = FAISS.from_documents(docs, embeddings)
db.save_local("faiss_index")
print("向量库构建完成，已保存到本地。")

/var/folders/3t/8_rvw5nj0gs2j9kwyv8n0mhr0000gp/T/ipykernel_90839/613233525.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")


向量库构建完成，已保存到本地。


# 5、创建AI问答链

In [5]:
# --- 第四步：创建AI问答链 ---
# 从 .env 文件中获取API密钥
api_key = os.getenv("DEEPSEEK_API_KEY")
if api_key is None:
    raise ValueError("请在 .env 文件中设置 DEEPSEEK_API_KEY")
    
os.environ["DEEPSEEK_API_KEY"] = api_key

# 初始化 DeepSeek 模型
llm = ChatDeepSeek(model="deepseek-chat")
# 将FAQ检索器和LLM组合成一个问答链
qa = RetrievalQA.from_chain_type(llm=llm, retriever=db.as_retriever())

# 6、开始提问

In [8]:
# --- 第五步：开始提问 ---
query = "什么是机器学习？"
try:
    result = qa.invoke(query)
    print("🔍 问题:", query)
    print("🤖 答案:", result["result"])
except Exception as e:
    msg = str(e)
    if "402" in msg and "Insufficient Balance" in msg:
        print("DeepSeek API 余额不足（402）。请充值或更换有余额的 API Key。")
    else:
        print("提问失败：", msg)

🔍 问题: 什么是机器学习？
🤖 答案: 机器学习是AI的一个子领域，它使计算机无需明确编程即可学习。
